In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# Load the Breast Cancer dataset
data = load_breast_cancer()

X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Number of classes:", len(np.unique(y)))
print("Classes:", data.target_names)

Dataset shape: (569, 30)
Number of classes: 2
Classes: ['malignant' 'benign']


In [4]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 455
Testing samples: 114


In [5]:
# Create the scaler
scaler = StandardScaler()

# IMPORTANT:
# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_scaled = scaler.transform(X_test)

print("Training data scaled successfully.")
print("Scaled shape:", X_train_scaled.shape)

Training data scaled successfully.
Scaled shape: (455, 30)


In [6]:
# Train Logistic Regression model
model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

print("Model training completed.")

Model training completed.


In [7]:
# Make predictions on test data
y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9825

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [8]:
# Create a directory to store trained artifacts
os.makedirs("model", exist_ok=True)

print("Model directory created.")

Model directory created.


In [9]:
# Save the trained model
joblib.dump(model, "model/model.pkl")

# Save the scaler
joblib.dump(scaler, "model/scaler.pkl")

print("Model saved as model/model.pkl")
print("Scaler saved as model/scaler.pkl")

Model saved as model/model.pkl
Scaler saved as model/scaler.pkl


In [10]:
# Load the saved model and scaler
loaded_model = joblib.load("model/model.pkl")
loaded_scaler = joblib.load("model/scaler.pkl")

print("Model loaded successfully.")
print("Scaler loaded successfully.")

Model loaded successfully.
Scaler loaded successfully.


In [11]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np

# Create FastAPI application
app = FastAPI(title="Breast Cancer Prediction API")

# Load trained model and scaler
model = joblib.load("model/model.pkl")
scaler = joblib.load("model/scaler.pkl")


# Define the request format
class PredictionRequest(BaseModel):
    features: list[float]


@app.get("/")
def home():
    return {
        "message": "ML Model API is running"
    }


@app.post("/predict")
def predict(request: PredictionRequest):

    # Convert input features to NumPy array
    features = np.array(request.features).reshape(1, -1)

    # Apply the saved scaler
    scaled_features = scaler.transform(features)

    # Generate prediction
    prediction = model.predict(scaled_features)[0]

    # Get prediction probability
    probability = model.predict_proba(scaled_features).max()

    return {
        "prediction": int(prediction),
        "confidence": round(float(probability), 4)
    }

Writing app.py


In [13]:
import subprocess
import sys
import time

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "app:app",
        "--host",
        "127.0.0.1",
        "--port",
        "8000"
    ]
)

time.sleep(3)

print("FastAPI server started.")
print("API URL: http://127.0.0.1:8000")
print("Swagger Docs: http://127.0.0.1:8000/docs")

FastAPI server started.
API URL: http://127.0.0.1:8000
Swagger Docs: http://127.0.0.1:8000/docs


In [14]:
import requests

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'ML Model API is running'}


In [15]:
# Take one sample from the test dataset
sample = X_test[0]

# Create request payload
payload = {
    "features": sample.tolist()
}

# Send POST request
response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=payload
)

print("Status code:", response.status_code)
print("Prediction response:", response.json())

Status code: 200
Prediction response: {'prediction': 0, 'confidence': 1.0}


In [16]:
# Display the files created for model serving
print("Saved files:")

for root, dirs, files in os.walk("model"):
    for file in files:
        print(os.path.join(root, file))

print("\nFastAPI application: app.py")

Saved files:
model/model.pkl
model/scaler.pkl

FastAPI application: app.py
